In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.coordinates as coord
import astropy.units as u
import emcee
import sys
import fitsio
from scipy.special import factorial
from astropy.table import Table
from numpy.polynomial.polynomial import Polynomial
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

if './SelfCalGroupFinder/py/' not in sys.path:
    sys.path.append('./SelfCalGroupFinder/py/')
from pyutils import *
import plotting as pp
from dataloc import *
from bgs_helpers import *
import catalog_definitions as cat
from groupcatalog import *

%load_ext autoreload
%autoreload 2

In [ ]:
#table = create_merged_file(BGS_Y3_ANY_FULL_FILE, IAN_BGS_Y3_MERGED_FILE_LOA, "3")
table = Table.read(IAN_BGS_Y3_MERGED_FILE_LOA)

In [ ]:
# Use a previous run of the group catalog, which has nice cuts applied to it, to get a clean sample of galaxies to do PCA on.
bgs_y3_pzp_2_6_c2 = deserialize(cat.bgs_y3_pzp_2_6_c2)
df = bgs_y3_pzp_2_6_c2.all_data
df = df.loc[df['Z_ASSIGNED_FLAG'] == AssignedRedshiftFlag.DESI_SPEC.value] # Only DESI observed ones of course
df = df.loc[:, ['TARGETID']] # Drop everything else from group catalog
print(len(df))

In [ ]:
names = [name for name in table.colnames if len(table[name].shape) <= 1] 
dft = table[names].to_pandas()

df = df.merge(dft, on='TARGETID', how='inner')
print(len(df))

In [ ]:
print(df.columns)

In [ ]:
# Derive some columns to include in PCA
df['G-R'] = df['ABSMAG01_SDSS_G'] - df['ABSMAG01_SDSS_R']
df['G-Z'] = df['ABSMAG01_SDSS_G'] - df['ABSMAG01_SDSS_Z']
df['R-Z'] = df['ABSMAG01_SDSS_R'] - df['ABSMAG01_SDSS_Z']
df['SSFR'] = df['SFR'] / np.power(10, df['LOGMSTAR'])
df['LOGSSFR'] = np.log10(df['SSFR'])

# Set inf to na
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [ ]:
# Plot LOGSSFR for QUIESCENT vs STAR-FORMING to see if it looks reasonable
red = df.loc[df['QUIESCENT']]
blue = df.loc[~df['QUIESCENT']] 
plt.hist(red['LOGSSFR'], bins=100, alpha=0.5, label='Quiescent')
plt.hist(blue['LOGSSFR'], bins=100, alpha=0.5, label='Star-forming')
plt.legend()
plt.title('LOGSSFR for Quiescent vs Star-forming')

In [ ]:
weird = blue.loc[blue['LOGSSFR'] < -15]
weird
# We may need to do something about SFR values if we want to include. 
# Not sure how bad data is, should investigate the claimed IVAR and talk to John.

In [ ]:
# Seperate into test/training sets
from sklearn.model_selection import train_test_split
train, test = train_test_split(df, test_size=0.1, random_state=894)

# TODO galaxy density in X mpc around each target
# TODO Concentration r90 / SHAPE_R. Need to integrate SERSIC to get r90. Or is having SERSIC and SHAPE_R enough?
# TODO Consider how to incorporate redshift 'Z'. Could do PCA on residuals after redshift trend removal?
# TODO get VDISP in fsf. Recalc on NERSC 
# TODO HBETA_EW could try
# TODO ZZSUN for metallicity.
feature_cols = ['ABSMAG01_SDSS_R', 'ABSMAG01_SDSS_G', 'ABSMAG01_SDSS_Z', 
                'LOGMSTAR', 'DN4000_MODEL', 'SHAPE_R', 'SERSIC', 'LOGSSFR']
                #'G-R', 'G-Z', 'R-Z'] # This is redundant. Fine to include, but complicates interpretation of loadings. Maybe add back in later after understanding the main features.
n_features = len(feature_cols)

# Drop rows with any NaN in the selected features
train_clean = train.dropna(subset=feature_cols)
test_clean = test.dropna(subset=feature_cols)
print(f"Training samples after NaN drop: {len(train_clean)} / {len(train)}")
print(f"Test samples after NaN drop: {len(test_clean)} / {len(test)}")

In [ ]:
# Scale first - critical since features have very different units/ranges
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_clean[feature_cols])
test_scaled = scaler.transform(test_clean[feature_cols])

In [ ]:
# Compare the original components to the scaled ones
for f in feature_cols:
    plt.hist(train_clean[f], bins=100, alpha=0.5, label=f'Original {f}')
    plt.hist(train_scaled[:, feature_cols.index(f)], bins=100, alpha=0.5, label=f'Scaled {f}')
    plt.legend()
    plt.title(f'Original vs Scaled {f}')
    plt.show()

In [ ]:
# Fit full PCA on training set
pca_full = PCA(n_components=n_features)
pca_full.fit(train_scaled)

# Explained variance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(1, n_features+1), pca_full.explained_variance_ratio_, color='steelblue')
axes[0].set_xlabel('PCA Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Variance per Component')

axes[1].plot(range(1, n_features+1), np.cumsum(pca_full.explained_variance_ratio_), 'ko-')
axes[1].axhline(0.95, color='red', linestyle='--', label='95%')
axes[1].axhline(0.99, color='orange', linestyle='--', label='99%')
axes[1].set_xlabel('Number of PCA Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:

# Autoencoder reconstruction MSE as a function of number of components kept
n_components_range = range(1, n_features + 1)
mse_train = []
mse_test = []

for n in n_components_range:
    pca_n = PCA(n_components=n)
    # Encode (project to n components) then decode (reconstruct)
    train_encoded = pca_n.fit_transform(train_scaled)
    train_reconstructed = pca_n.inverse_transform(train_encoded)
    test_encoded = pca_n.transform(test_scaled)
    test_reconstructed = pca_n.inverse_transform(test_encoded)

    mse_train.append(mean_squared_error(train_scaled, train_reconstructed))
    mse_test.append(mean_squared_error(test_scaled, test_reconstructed))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(n_components_range, mse_train, 'ko-', label='Train')
ax.plot(n_components_range, mse_test, 'rs-', label='Test')
ax.set_xlabel('Number of PCA Components Kept')
ax.set_ylabel('Reconstruction MSE (standardized units)')
ax.set_title('PCA Autoencoder Performance')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print summary table
print(f"\n{'N Components':<15} {'Cumul. Var':<15} {'Train MSE':<15} {'Test MSE':<15}")
print("-" * 60)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
for i, n in enumerate(n_components_range):
    print(f"{n:<15} {cumvar[i]:<15.4f} {mse_train[i]:<15.4f} {mse_test[i]:<15.4f}")

In [ ]:
# Print feature loadings for top components
print("\nFeature loadings for top 6 components:")
loadings = pd.DataFrame(
    pca_full.components_[:6].T,
    index=feature_cols,
    columns=[f'PC{i+1}' for i in range(6)]
)
print(loadings.round(3).to_string())